In [ ]:
import glob

import xarray as xr

da_instant = glob.glob("../data/raw/**/*instant.nc")
da_accum = glob.glob("../data/raw/**/*accum.nc")

print(da_accum)
print(da_instant)

In [ ]:
da_instant = xr.open_mfdataset(da_instant, chunks='auto')
da_accum = xr.open_mfdataset(da_accum, chunks='auto')

In [ ]:
# checking experiment version (expver)
df_instant_expver = da_instant.expver.to_dataframe()
df_accum_expver = da_accum.expver.to_dataframe()

print(df_instant_expver.head())
print(df_accum_expver.head())

# check unqiue values for expver
df_instant_expver.expver.nunique()

In [ ]:
df_accum_expver.expver.nunique()

Its safe to drop expver here, since the values are all `1`.

In [ ]:
da_instant = da_instant.drop_vars("expver")
da_accum = da_accum.drop_vars("expver")

Transform `total precipitation`, as it is accumulated

In [ ]:

tp_daily = da_accum.resample(valid_time="1D").last()

In [ ]:
tp_daily

Resample instant variables also

In [ ]:
instant_daily = da_instant.resample(valid_time="1D").mean()
instant_daily

In [ ]:
ds_daily = xr.merge([
    instant_daily,
    tp_daily,
])

In [ ]:
ds_daily

In [ ]:
import xarray as xr

ds = xr.open_dataset("../data/interim/combined_2020-2024_daily.nc", chunks="auto")
ds.dims

In [ ]:
ds.drop_vars("number")

In [ ]:
combined_df = ds.to_dataframe()

In [ ]:
del combined_df["number"]

In [ ]:
combined_df.vars

In [ ]:
combined_df.to_csv("combined_2020-2024_daily.csv")

In [ ]:
from scripts import Operations
from constants import DATA_CONFIG

raw_dir: str =  "../" + DATA_CONFIG["paths"]["raw"]
export_dir: str = "../" + DATA_CONFIG["paths"]["interim"]

merged_dataset = (
    Operations(raw_dir=raw_dir)
    .deserialize_accum(drop_expver=True, drop_number=True)
    .deserialize_instant(drop_expver=True, drop_number=True)
    .resample_accum()
    .resample_instant()
    .merge()
    .apply()
)

merged_dataset.serialize(dir=f"{export_dir}/export_default.nc", complevel=9)